# Classificação conservadora de documentos judiciais brasileiros

**Objetivo:** classificar a espécie do documento principal, não o assunto ou conteúdo citado. Abrangência pretendida: variações de redação e ritos das 27 UFs, Justiças Estadual, Federal e do Trabalho; **não há garantia de cobertura de todas as práticas locais sem validação empírica**.

**Taxonomia:** `peticao_inicial`, `sentenca`, `acordao`, `obrigacao_de_fazer`, `laudo`, `extrato`, `outros_documentos`.

**Política de risco:** preferir abstenção/revisão a forçar rótulo. `outros_documentos` mistura classe residual e abstenção; consulte sempre `status`/`requires_review` para distingui-las. PDF digitalizado sem camada textual requer OCR **em uma etapa separada**, antes de voltar a classificar.

**Segurança:** use arquivos locais autorizados, idealmente sintéticos/anonimizados. Não faça upload externo sem aprovação jurídica, de segurança e DPO. Logs omitem texto extraído e credenciais. Registre versão das regras/prompts, amostras de validação por tribunal e matrices de confusão antes de uso produtivo.

## Notebook 1 — Motor de regex
Regras cumulativas específicas; evidências em famílias independentes. Regex de cabeçalho ou termos isolados não decide uma classe.

In [ ]:
%pip install "PyMuPDF==1.26.7"

### Como ler a implementação

A sequência é: **validar o caminho → abrir PDF → extrair página a página → verificar se há texto suficiente → classificar → devolver evidências ou solicitar revisão**.

As linhas iniciadas por `#` explicam **por que cada instrução existe**, além de entradas e saídas. `PdfText` guarda a extração; `normalize` padroniza palavras; `extract_pdf` lê PDFs; `load_paths` localiza os arquivos. Os limites de páginas/caracteres previnem consumo excessivo, porém podem deixar um documento incompleto. O texto do processo não é impresso no log.

**Conceitos básicos:** `def` cria uma função; `return` entrega o resultado; `if` verifica uma condição; `for` percorre itens; `try/except` trata falhas explicitamente; `dict` reúne campos identificados por nomes.


In [ ]:
from __future__ import annotations
import json
import logging
import os
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Any
import fitz

# Os identificadores técnicos são intencionalmente registrados sem conteúdo do processo.
# Configura mensagens de execução sem registrar o texto dos processos, que pode conter dados pessoais.
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
# Cria um identificador para localizar eventos deste classificador nos registros operacionais.
logger = logging.getLogger("judicial_classifier")
# Define as únicas categorias aceitas. A grafia fixa facilita integrações e evita rótulos inconsistentes.
LABELS = ("peticao_inicial", "sentenca", "acordao", "obrigacao_de_fazer", "laudo", "extrato", "outros_documentos")

# Agrupa os resultados da extração em um objeto imutável para impedir alterações acidentais entre etapas.
@dataclass(frozen=True)
class PdfText:
    # Caminho do PDF de origem; não é utilizado como evidência para classificar.
    path: str
    # Texto das primeiras páginas, priorizado porque costuma conter o título da peça.
    first_pages: str
    # Texto das páginas efetivamente lidas, útil para análises e auditoria controlada.
    full_text: str
    # Quantidade total de páginas do arquivo original.
    page_count: int
    # Quantidade de caracteres extraídos; ajuda a identificar extrações pobres.
    char_count: int
    # Número de páginas que retornaram algum texto extraível.
    pages_with_text: int
    # Situação da leitura: texto suficiente ou documento que precisa de tratamento/revisão.
    status: str


def normalize(text: str) -> str:
    """Remove variações de acentos e espaços, preservando o conteúdo lógico."""
    # Converte letras a uma forma comparável, inclusive quando há variações Unicode.
    text = unicodedata.normalize("NFKD", text.casefold())
    # Elimina marcas de acento para que o mesmo padrão funcione com e sem acentuação.
    text = "".join(c for c in text if not unicodedata.combining(c))
    # Une espaços e quebras para facilitar buscas por termos no texto corrido.
    return re.sub(r"\s+", " ", text).strip()


def extract_pdf(path: str | Path, *, max_pages: int = 200, head_pages: int = 3,
                max_chars: int = 200_000, min_chars: int = 100) -> PdfText:
    """Extrai páginas em ordem, com limites explícitos de recursos e detecção de texto insuficiente."""
    # Converte o caminho recebido em um objeto que facilita validar arquivo e extensão.
    source = Path(path)
    # Rejeita entradas inexistentes e formatos diferentes de PDF antes de abrir arquivos.
    if not source.is_file() or source.suffix.lower() != ".pdf":
        raise ValueError("Informe um arquivo PDF existente.")
    # Rejeita limites inválidos para evitar leituras ilimitadas ou resultados incoerentes.
    if max_pages < 1 or head_pages < 1 or max_chars < 100 or min_chars < 0:
        raise ValueError("Parâmetros de extração inválidos.")
    # Inicia bloco protegido para diferenciar falha de arquivo de erro de classificação.
    try:
        # Abre o PDF pelo PyMuPDF e fecha o recurso automaticamente ao final.
        with fitz.open(source) as doc:
            # Impede classificar documento criptografado sem acesso autorizado.
            if doc.needs_pass:
                raise ValueError("PDF protegido por senha; forneça arquivo autorizado e desbloqueado.")
            # Identifica arquivos sem páginas, que não têm conteúdo para análise.
            if len(doc) == 0:
                raise ValueError("PDF sem páginas.")
            # Reserva a lista que guarda o texto na sequência original das páginas.
            texts = []
            # Inicia a contagem de páginas com texto detectável.
            count = 0
            # Inicia o controle do limite de caracteres lidos.
            total = 0
            # Lê as páginas em ordem e respeita o máximo configurado.
            for index in range(min(len(doc), max_pages)):
                # Solicita texto ao PyMuPDF; sort=True tenta aproximar a ordem natural de leitura.
                page_text = doc[index].get_text("text", sort=True)
                # Verifica se a página possui algo além de espaços.
                if page_text.strip():
                    # Conta uma página como textual somente quando foi encontrado conteúdo.
                    count += 1
                # O separador evita unir palavras de páginas distintas.
                # Calcula quantos caracteres ainda podem ser incluídos sem exceder o limite.
                remaining = max_chars - total
                # Encerra a extração quando o limite de segurança foi atingido.
                if remaining <= 0:
                    # Para o laço; as páginas restantes não entram no texto analisado.
                    break
                # Corta somente o excedente para respeitar o teto de leitura.
                page_text = page_text[:remaining]
                # Guarda o texto desta página na lista de páginas já processadas.
                texts.append(page_text)
                # Atualiza o consumo total de caracteres.
                total += len(page_text)
            # Monta o bloco das páginas iniciais e insere separadores entre páginas.
            first = "\n[PAGE]\n".join(texts[:head_pages])
            # Monta todo o texto lido, preservando divisões de páginas.
            full = "\n[PAGE]\n".join(texts)
            # Se não houver conteúdo mínimo, pede revisão; PDF digitalizado pode exigir OCR.
            status = "ok" if len(normalize(full)) >= min_chars else "texto_insuficiente_ou_pdf_digitalizado"
            # Avisa que houve leitura parcial, pois isso pode afetar a conclusão.
            if len(doc) > max_pages or total >= max_chars:
                logger.warning("Extração limitada pelo teto configurado; resultado pode exigir revisão.")
            # Entrega texto, contagens e situação da extração em um único objeto.
            return PdfText(str(source), first, full, len(doc), len(full), count, status)
    # Transforma erros de arquivo do PyMuPDF em mensagem de negócio compreensível.
    except (fitz.FileDataError, fitz.EmptyFileError) as exc:
        raise ValueError("PDF inválido ou corrompido.") from exc


def load_paths(directory: str | Path) -> list[Path]:
    """Lista apenas PDFs de um diretório; não percorre pastas ocultas por padrão."""
    # Identifica a pasta na qual serão procurados os PDFs.
    root = Path(directory)
    # Evita continuar se a pasta não existir.
    if not root.is_dir():
        raise ValueError("Diretório de PDFs inexistente.")
    # Lista PDFs ordenados; não lê pastas internas e torna a execução previsível.
    return sorted(x for x in root.glob("*.pdf") if x.is_file())


### Leitura guiada das regras

- `PATTERNS` é o dicionário central de categorias. Cada categoria tem **famílias independentes** (por exemplo, título, dispositivo e autoridade). As expressões com `r"..."` procuram formas alternativas de escrever um termo; `\b` delimita palavras; `(?:a|b)` aceita alternativas; `\s+` aceita espaços; `(?:^|\n)` exige começo de texto/linha.
- `NEGATIVE_TITLES` interrompe a tentativa quando o documento principal apresenta título forte de outra espécie (contestação, recurso, despacho etc.).
- `signals` registra padrões encontrados; `valid` só aceita uma classe se **todas as famílias obrigatórias** existirem; `candidates` lista hipóteses aprovadas.
- Em caso de nenhum candidato, dois candidatos, título fora do cabeçalho ou indício de jurisprudência transcrita, retorna **outros_documentos** com motivo de revisão; isso é proposital para reduzir falsos positivos.
- A correspondência de regex é **heurística**, não uma comprovação jurídica da espécie da peça. PDFs com peças concatenadas exigem atenção especial.


In [ ]:
# Versão das regras: alterar ao modificar critérios e repetir validação por tribunal/UF.
# Versão das regras: alterar quando um padrão, bloqueio ou critério de decisão mudar.
RULE_VERSION = "2.0.0"

# Os conjuntos representam FAMÍLIAS de sinais; vários sinônimos da mesma família
# não contam como evidências independentes.
# Organiza expressões por categoria e por família de evidências, sem somar sinônimos da mesma família.
PATTERNS = {
 "peticao_inicial": {
  "address": [r"\b(?:excelentissim[oa]|ilustrissim[oa])\s+(?:senhor(?:a)?\s+)?(?:doutor(?:a)?\s+)?(?:juiz(?:a)?|desembargador(?:a)?)\b", r"\b(?:ao|a|perante\s+o)\s+(?:douto\s+)?(?:juizo|juiz(?:a)?)\s+(?:de\s+direito\s+)?(?:da|do)\s+(?:\d{1,3}\s*[aºªo]?\s+)?(?:vara|juizado|comarca|subsecao|secao\s+judiciaria)\b",r"\b(?:ao|a)\s+juizo\s+(?:da|do)\s+(?:vara|juizado|comarca)\b"],
  "filing": [r"\b(?:ajuiza|propoe|vem\s+propor|vem\s+ajuizar|vem\s+interpor)\s+(?:a\s+presente\s+)?(?:acao|reclamacao\s+trabalhista|demanda)\b",r"\b(?:acao\s+(?:de\s+)?(?:indenizacao|reparacao|cobranca|consignacao|prestacao\s+de\s+contas|revisional|anulatoria|declaratoria|monitoria|popular|civil\s+publica|possessoria|de\s+obrigacao\s+de\s+fazer)|reclamacao\s+trabalhista|mandado\s+de\s+seguranca|habeas\s+data|peticao\s+inicial)\b"],
  "requests": [r"\b(?:requer|requerem|postula|pleiteia)\s+(?:a\s+)?(?:citacao|notificacao|intimacao)\s+(?:do|da|dos|das)\s+(?:reu|requerid[oa]|reclamad[oa]|parte\s+contraria)\b",r"\b(?:dos\s+pedidos|pedidos\s+finais|requerimentos\s+finais|do\s+valor\s+da\s+causa|da\s+causa\s+de\s+pedir)\b",r"\b(?:a\s+presente\s+acao|a\s+presente\s+demanda)\s+(?:tem\s+por\s+objeto|visa)\b"]},
 "sentenca": {
  "identity": [r"(?:^|\n)\s*(?:poder\s+judiciario\s*)?(?:sentenca|sentenca\s+homologatoria|sentenca\s+de\s+merito)\s*(?:\n|$|[-:])",r"\b(?:relatorio\s+dispensado\s+nos\s+termos|vistos\s+e\s+examinados)\b"],
  "disposition": [r"\b(?:julgo\s+(?:o\s+pedido\s+)?(?:procedente|improcedente|parcialmente\s+procedente)|julgo\s+extinto|extingo\s+(?:o\s+)?(?:feito|processo)|homologo\s+(?:o\s+acordo|a\s+transacao|por\s+sentenca)|condeno\s+(?:a\s+)?(?:parte|re|reu|reclamad[oa])|resolvo\s+o\s+merito)\b",r"\b(?:com\s+(?:fundamento|fulcro)\s+no\s+art(?:igo)?\.?\s*\d+\s+(?:do|da)\s+cpc|sem\s+resolucao\s+do\s+merito|com\s+resolucao\s+do\s+merito)\b"],
  "authority": [r"\bjuiz(?:a)?\s+(?:de\s+direito|federal|do\s+trabalho|leig[oa])\b",r"\b(?:publique-se\s*\.?\s*registre-se|registre-se\s*\.?\s*intimem-se|p\.?\s*r\.?\s*i\.?)\b"]},
 "acordao": {
  "identity": [r"(?:^|\n)\s*(?:acordao|ementa\s+e\s+acordao)\s*(?:\n|$|[:.-])",r"\b(?:ementa\s*[:\-]|inteiro\s+teor\s+do\s+acordao)\b"],
  "collegial": [r"\b(?:acordam\s+(?:os|as|a\s+turma|os\s+desembargadores|os\s+ministros)|decidem\s+(?:os|as)\s+(?:desembargadores|ministros|integrantes))\b",r"\b(?:por\s+(?:unanimidade|maioria)\s+(?:de\s+votos|dos\s+votos)|a\s+turma\s+(?:decidiu|resolveu)|o\s+colegiado\s+(?:decidiu|deliberou))\b"],
  "panel": [r"\b(?:desembargador(?:a)?|ministr[oa])\s+relator(?:a)?\b",r"\b(?:relator(?:a)?|redator(?:a)?\s+designad[oa])\b",r"\b(?:\d{1,2}\s*[aªo]?\s+turma|\d{1,2}\s*[aªo]?\s+camara|turma\s+recursal|camara\s+(?:civel|criminal)|secao\s+especializada|orgao\s+especial)\b"]},
 "obrigacao_de_fazer": {
  "identity": [r"(?:^|\n)\s*(?:termo|certidao|comprovante|declaracao|relatorio|atestado)\s+(?:de\s+)?(?:cumprimento|execucao|efetivacao|atendimento)\s+(?:da\s+|de\s+)?obrigacao\s+de\s+fazer\b",r"(?:^|\n)\s*(?:comprovacao|certificacao)\s+de\s+(?:cumprimento|efetivacao)\s+de\s+(?:ordem|determinacao)\s+judicial\b"],
  "completion": [r"\b(?:obrigacao\s+de\s+fazer\s+(?:cumprida|efetivada|atendida|executada)|determinacao\s+judicial\s+(?:cumprida|atendida|executada)|providencia\s+(?:efetivada|comprovadamente\s+realizada))\b",r"\b(?:certifico\s+(?:que\s+)?(?:foi|ter\s+sido)\s+(?:cumprid[oa]|efetivad[oa])|comprova(?:-se)?\s+o\s+(?:integral\s+)?cumprimento)\b"]},
 "laudo": {
  "identity": [r"(?:^|\n)\s*(?:laudo|relatorio)\s+(?:pericial|medico(?:-pericial)?|odontologico|psicologico|psiquiatrico|contabil|tecnico|de\s+engenharia|grafotecnico|documentoscopico|de\s+insalubridade|de\s+periculosidade|social|de\s+avaliacao)\b",r"(?:^|\n)\s*laudo\s+de\s+pericia\b"],
  "method": [r"\b(?:objeto\s+da\s+pericia|metodologia\s+(?:pericial|empregada|utilizada)|procedimentos\s+periciais|exame\s+pericial|diligencia\s+pericial)\b",r"\b(?:respostas?\s+aos\s+quesitos|quesitos\s+(?:do\s+juizo|das\s+partes)|conclusao\s+(?:pericial|tecnica)|analise\s+tecnica\s+dos\s+elementos)\b"],
  "author": [r"\b(?:perito\s+(?:judicial|nomeado|do\s+juizo)|perita\s+(?:judicial|nomeada|do\s+juizo)|responsavel\s+tecnico|assistente\s+tecnico)\b",r"\b(?:crea|crm|crc|crp)\s*[-:/]?\s*[a-z]{0,3}\s*\d{3,}\b"]},
 "extrato": {
  "identity": [r"(?:^|\n)\s*extrato\s+(?:bancario|de\s+conta(?:\s+corrente)?|de\s+movimentacao|de\s+lancamentos|de\s+pagamentos?|de\s+beneficio|de\s+fgts|analitico|previdenciario|de\s+poupanca|de\s+emprestimo|de\s+financiamento|de\s+cartao)\b",r"(?:^|\n)\s*(?:demonstrativo\s+de\s+movimentacao\s+(?:financeira|bancaria)|historico\s+de\s+transacoes)\b"],
  "balances": [r"\b(?:saldo\s+(?:anterior|inicial|final|atual|disponivel|bloqueado)|saldo\s+em\s+conta|saldo\s+do\s+periodo)\b",r"\b(?:total\s+de\s+creditos|total\s+de\s+debitos|saldo\s+apos\s+lancamento)\b"],
  "transactions": [r"\b(?:data\s+(?:do\s+)?lancamento|data\s+historico\s+valor|data\s+descricao\s+valor|debito\s+credito\s+saldo|credito\s+debito\s+saldo|movimentacoes\s+do\s+periodo|historico\s+de\s+movimentacoes)\b",r"\b(?:lancamentos\s+(?:da\s+conta|do\s+periodo)|transacoes\s+realizadas|movimentacao\s+financeira)\b"]},
}

# Bloqueios são avaliados NO COMEÇO do arquivo, antes de qualquer pontuação.
# Não usar termos de jurisprudência como evidência do documento principal.
# Lista títulos de peças fora do escopo. Um título forte no começo impede falso positivo posterior.
NEGATIVE_TITLES = [
 r"(?:^|\n)\s*(?:contestacao|reconvencao|replica|impugnacao\s+(?:ao\s+laudo|a\s+contestacao)|embargos?\s+(?:de\s+declaracao|a\s+execucao|de\s+terceiro)|contrarrazoes|razoes\s+(?:de\s+apelacao|recursais)|apelacao|agravo\s+(?:de\s+instrumento|interno)|recurso\s+(?:ordinario|especial|extraordinario|inominado)|manifestacao|aditamento\s+a\s+inicial|cumprimento\s+de\s+sentenca|execucao\s+de\s+titulo|peticao\s+intermediaria|procuracao|substabelecimento|certidao\s+de\s+intimacao|decisao\s+(?:interlocutoria|monocratica)|despacho|oficio|mandado\s+judicial)\b"
]
# Detecta sinais de que sentença ou acórdão podem estar apenas transcritos em outra peça.
QUOTATION_MARKERS = [r"\b(?:jurisprudencia|precedentes\s+jurisprudenciais|colaciona-se\s+o\s+seguinte\s+acordao|conforme\s+sentenca\s+transcrita|transcricao\s+do\s+acordao|trecho\s+da\s+sentenca)\b"]


def normalize_lines(text: str) -> str:
    """Preserva títulos e fronteiras de linha após normalizar acentuação."""
    # Normaliza cada linha individualmente, preservando limites importantes para títulos.
    return "\n".join(normalize(line) for line in text.splitlines() if line.strip())


def matched(patterns: list[str], text: str) -> list[str]:
    """Retorna somente padrões identificados, sem capturar o texto sensível."""
    # Retorna padrões encontrados, nunca trechos do processo, para reduzir exposição de dados.
    return [pattern for pattern in patterns if re.search(pattern, text, flags=re.IGNORECASE)]


def classify_regex(pdf: PdfText) -> dict[str, Any]:
    """Exige famílias independentes e se abstém diante de ambiguidade estrutural."""
    # Cria a estrutura mínima de resposta comum a classificações e abstenções.
    base = {"rule_version": RULE_VERSION, "candidates": [], "evidence": {}}
    # Não classifica PDFs sem texto suficiente; pode ser imagem ou extração incompleta.
    if pdf.status != "ok":
        return {**base, "label": "outros_documentos", "status": "revisao_manual", "reason": pdf.status}
    # O início concentra cabeçalho e identidade. Trechos distantes são intencionalmente
    # excluídos da decisão para reduzir contaminação por anexos e jurisprudência.
    # Normaliza primeiras páginas sem perder a separação das linhas de cabeçalho.
    lines = normalize_lines(pdf.first_pages)
    # Usa trecho inicial relativamente amplo para buscar diferentes famílias de sinais.
    head = "\n" + lines[:14000]
    # Usa trecho ainda menor para verificar títulos de peças incompatíveis com as classes-alvo.
    earliest = "\n" + lines[:1800]
    # Dá prioridade à identidade negativa da peça antes de procurar sinais positivos.
    if matched(NEGATIVE_TITLES, earliest):
        return {**base, "label": "outros_documentos", "status": "sem_regra_conclusiva", "reason": "titulo_de_outra_especie"}
    # Para cada classe, registra quais famílias de regras foram encontradas no texto inicial.
    signals = {label: {group: matched(patterns, head) for group, patterns in groups.items()}
               for label, groups in PATTERNS.items()}
    # A presença de um título costuma ser exigida nas classes cuja identificação
    # textual não é confiável por palavras soltas.
    # Aplica combinações obrigatórias; um termo isolado não satisfaz uma classe.
    valid = {
      "peticao_inicial": all(signals["peticao_inicial"][g] for g in ("address", "filing", "requests")),
      "sentenca": all(signals["sentenca"][g] for g in ("identity", "disposition", "authority")),
      "acordao": all(signals["acordao"][g] for g in ("identity", "collegial", "panel")),
      "obrigacao_de_fazer": all(signals["obrigacao_de_fazer"][g] for g in ("identity", "completion")),
      "laudo": bool(signals["laudo"]["identity"] and signals["laudo"]["method"] and signals["laudo"]["author"]),
      "extrato": bool(signals["extrato"]["identity"] and signals["extrato"]["balances"] and signals["extrato"]["transactions"]),
    }
    # Guarda apenas as categorias que passaram por todos os critérios exigidos.
    candidates = [label for label, passed in valid.items() if passed]
    # Retém evidências das hipóteses encontradas para permitir análise de ambiguidades.
    available = {k: v for k, v in signals.items() if any(v.values())}
    # Se há zero ou múltiplos candidatos, abstém-se em vez de escolher por suposição.
    if len(candidates) != 1:
        return {**base, "label": "outros_documentos", "status": "revisao_manual" if candidates else "sem_regra_conclusiva",
                "reason": "categorias_conflitantes" if candidates else "evidencia_insuficiente",
                "candidates": candidates, "evidence": available}
    # Citações em peça principal podem satisfazer várias famílias: exigimos que
    # o título distintivo de laudo, extrato, decisão ou cumprimento apareça cedo.
    # Obtém a categoria única que passou por todas as famílias de evidências.
    label = candidates[0]
    # Categorias mais confundíveis exigem título identificador muito perto do começo.
    if label in ("sentenca", "acordao", "obrigacao_de_fazer", "laudo", "extrato"):
        # Indica a família de padrões que comprova a identidade documental.
        distinct = "identity"
        # Busca o título verdadeiro apenas na região inicial, evitando títulos dentro de anexos.
        if not any(re.search(pattern, "\n" + lines[:3500], flags=re.I)
                   for pattern in PATTERNS[label][distinct]):
            return {**base, "label": "outros_documentos", "status": "revisao_manual", "reason": "identidade_fora_do_cabecalho", "candidates": candidates, "evidence": available}
    # Sentenças e acórdãos citados em uma petição precisam de revisão se houver
    # indícios fortes de transcrição no cabeçalho.
    # Evita tratar jurisprudência ou citações reproduzidas como pronunciamento principal.
    if label in ("sentenca", "acordao") and matched(QUOTATION_MARKERS, earliest):
        return {**base, "label": "outros_documentos", "status": "revisao_manual", "reason": "possivel_transcricao", "candidates": candidates, "evidence": available}
    # Retorna classe, versão das regras e padrões responsáveis pela decisão.
    return {**base, "label": label, "status": "classificado_por_regras", "reason": "familias_independentes_confirmadas",
            "candidates": candidates, "evidence": signals[label]}


def classify_directory(directory: str | Path) -> list[dict[str, Any]]:
    """Processa cada PDF separadamente e registra erros sem vazar conteúdo."""
    # Prepara a lista de resultados: cada PDF será classificado de forma independente.
    results = []
    # Percorre os PDFs encontrados na pasta definida pelo usuário.
    for path in load_paths(directory):
        try:
            # Primeiro extrai o texto; somente depois executa o classificador.
            pdf = extract_pdf(path)
            # Adiciona classificação e contagem de páginas sem copiar o conteúdo extraído.
            results.append({"file": path.name, **classify_regex(pdf), "pages": pdf.page_count})
        # Isola falhas de um arquivo para que os próximos continuem sendo processados.
        except (ValueError, OSError) as exc:
            # Registra o tipo da falha, sem imprimir o texto do documento.
            logger.error("Falha na leitura do PDF %s: %s", path.name, type(exc).__name__)
            # Marca erro separadamente de classificação válida para revisão operacional.
            results.append({"file": path.name, "label": "outros_documentos", "status": "erro", "reason": type(exc).__name__})
    # Devolve todos os resultados para exportação ou inspeção posterior.
    return results


## Execução
Configure `PDF_DIR` para um diretório local contendo PDFs. Os resultados não contêm texto de documentos.

In [ ]:
PDF_DIR = "./pdfs"
# Após criar a pasta, execute:
# results = classify_directory(PDF_DIR)
# print(json.dumps(results, indent=2, ensure_ascii=False))

## Testes unitários sintéticos (não equivalem à validação em documentos reais)

In [ ]:
def synthetic_pdf(text: str) -> PdfText:
    """Cria objeto de teste sem usar documentos reais."""
    return PdfText("synthetic.pdf", text, text, 1, len(text), 1, "ok")

cases = [
("sentenca", "SENTENÇA\nVistos. Ante o exposto julgo procedente o pedido. Juiz de Direito. Publique-se. Registre-se."),
("acordao", "ACÓRDÃO\nEMENTA: Apelação. Relator. ACORDAM os desembargadores por unanimidade de votos. Câmara Cível."),
("peticao_inicial", "Excelentíssimo Senhor Doutor Juiz de Direito. Ação de indenização. Vem propor a presente ação. Dos pedidos. Requer a citação do réu."),
("laudo", "LAUDO PERICIAL\nObjeto da perícia. Metodologia pericial. Quesitos do juízo. Conclusão técnica. Perito judicial."),
("extrato", "EXTRATO BANCÁRIO\nSaldo anterior. Movimentações do período. Crédito débito saldo. Saldo final."),
("obrigacao_de_fazer", "Termo de cumprimento de obrigação de fazer\nObrigação de fazer cumprida. Determinação judicial cumprida."),
("outros_documentos", "CONTESTAÇÃO\nA parte ré contesta os pedidos. A petição inicial cita sentença e acórdão."),
("outros_documentos", "APELAÇÃO\nEmenta: ACÓRDÃO\nRelator. Acordam os desembargadores por unanimidade de votos."),
("outros_documentos", "DECISÃO MONOCRÁTICA\nACÓRDÃO\nRelator. Acordam os desembargadores por unanimidade de votos."),
("outros_documentos", "IMPUGNAÇÃO AO LAUDO\nLAUDO PERICIAL\nObjeto da perícia. Metodologia pericial. Perito judicial."),
("outros_documentos", "CUMPRIMENTO DE SENTENÇA\nTermo de cumprimento de obrigação de fazer. Obrigação de fazer cumprida."),
("outros_documentos", "PETIÇÃO DE JUNTADA\nSegue extrato bancário. Saldo inicial. Crédito débito saldo."),
("outros_documentos", "PLANILHA DE CÁLCULO\nValor principal. Correção monetária. Juros legais."),
("outros_documentos", "ACÓRDÃO\nRELATOR\nFoi citado por uma das partes, sem deliberação colegiada."),
("outros_documentos", "SENTENÇA\nO processo prossegue. Ausente dispositivo de julgamento."),
]
for expected, text in cases:
    result = classify_regex(synthetic_pdf(text))
    assert result["label"] == expected, (expected, result["label"], result["reason"])
print(f"Testes sintéticos aprovados: {len(cases)}")

## Limitações e validação
Extratos podem perder layout tabular; petições que incorporam decisões/peças e decisões sem títulos podem ficar sem classificação. Regras deliberadamente conservadoras reduzem algumas fontes de falso positivo, porém **recall e precisão não foram estimados**. Para validar: conjunto estratificado por UF, tribunal, classe, digitalização e compilação; dupla rotulagem humana, matriz de confusão, precisão por classe, taxa de abstenção e auditoria de falsos positivos. Versione mudanças antes de recalibrar regras.